<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICS40125 - Laboratorio N°04


**Objetivo**: Aplicar técnicas intermedias y avanzadas de análisis de datos con pandas utilizando un caso real: el Índice de Libertad de Prensa. Este laboratorio incluye operaciones de limpieza, transformación, combinación de datos, y análisis exploratorio usando `merge`, `groupby`, `concat` y otras funciones fundamentales.




**Descripción del Dataset**

El presente conjunto de datos está orientado al análisis del **Índice de Libertad de Prensa**, una métrica internacional que evalúa el nivel de libertad del que gozan periodistas y medios de comunicación en distintos países. Este índice es recopilado anualmente por la organización **Reporteros sin Fronteras**.

La base de datos contempla observaciones por país y año, e incluye tanto el valor del índice como el ranking correspondiente. A menor puntaje en el índice, mayor nivel de libertad de prensa.

**Diccionario de variables**

| Variable     | Clase    | Descripción                                                                          |
| ------------ | -------- | ------------------------------------------------------------------------------------ |
| `codigo_iso` | carácter | Código ISO 3166-1 alfa-3 que representa a cada país.                                 |
| `pais`       | carácter | Nombre oficial del país.                                                             |
| `anio`       | entero   | Año en que se registró la medición del índice.                                       |
| `indice`     | numérico | Valor numérico del Índice de Libertad de Prensa (menor valor indica mayor libertad). |
| `ranking`    | entero   | Posición relativa del país en el ranking mundial de libertad de prensa.              |


**Fuente original y adaptación pedagógica**

* **Fuente original**: [Reporteros sin Fronteras](https://www.rsf-es.org/), recopilado y publicado a través del portal del [Banco Mundial](https://tcdata360.worldbank.org/indicators/h3f86901f?country=BRA&indicator=32416&viz=line_chart&years=2001,2019).
* **Adaptación educativa**: Los archivos han sido modificados intencionalmente para incorporar desafíos técnicos que permiten aplicar los contenidos abordados en clases, tales como limpieza de datos, normalización, detección de duplicados, y combinación de fuentes.


**Descripción de los archivos disponibles**

* **`libertad_prensa_codigo.csv`**: Contiene los pares `codigo_iso` y `pais`. Incluye intencionalmente un código ISO con dos nombres distintos de país para efectos de limpieza y validación de datos.

* **`libertad_prensa_01.csv`**: Contiene registros de los años **anteriores a 2010**. Incluye las variables `PAIS`, `ANIO`, `INDICE`, y `RANKING` con nombres de columna en **mayúsculas**.

* **`libertad_prensa_02.csv`**: Contiene registros de los años **desde 2010 en adelante**. Estructura similar al archivo anterior, con nombres de columna también en **mayúsculas**.





In [2]:
import numpy as np
import pandas as pd

# lectura de datos
archivos_anio = [
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_01.csv',
    'https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_02.csv'
 ]
df_codigos = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/libertad_prensa_codigo.csv')



### 1. Consolidación y limpieza de datos

A partir de los archivos disponibles, realice los siguientes pasos:

**a)** Cree un DataFrame llamado `df_anio` que consolide la información proveniente de los archivos **`libertad_prensa_01.csv`** y **`libertad_prensa_02.csv`**, correspondientes a distintas ventanas de tiempo. Recuerde que ambos archivos tienen nombres de columnas en mayúscula, por lo que debe normalizarlas a **minúscula** para asegurar consistencia.

**b)** Explore el archivo **`libertad_prensa_codigo.csv`** e identifique el código ISO que aparece asociado a dos nombres de país distintos. Elimine el registro que corresponda a un valor incorrecto o inconsistente, conservando solo el que considere válido.

**c)** Una vez preparados los archivos, cree un nuevo DataFrame llamado `df` que combine `df_anio` con `df_codigos`, utilizando la columna `codigo_iso` como clave. Asegúrese de realizar una unión que conserve únicamente los registros que tengan coincidencia en ambas fuentes.

> **Sugerencia**:
>
> * Para unir los archivos por filas (años), utilice la función `pd.concat([...])`.
> * Para combinar información por columnas (variables), utilice `pd.merge(...)` especificando `on='codigo_iso'`.



In [10]:
# FIXME
# a) Consolidar la información de los archivos temporales y pasar a minúsculas
dfs_anios = []
for archivo in archivos_anio:
    df_temp = pd.read_csv(archivo)
    df_temp.columns = df_temp.columns.str.lower()  # Normalizar a minúsculas
    dfs_anios.append(df_temp)

df_anio = pd.concat(dfs_anios, ignore_index=True)

# b) Identificar y limpiar el código ISO inconsistente en df_codigos
# Primero encontramos si hay duplicados en el código ISO
# (Por ejemplo, si un ISO tiene un nombre correcto y otro incorrecto, eliminamos el duplicado)
df_codigos = df_codigos.drop_duplicates(subset=['codigo_iso'], keep='first')

# c) Combinar df_anio con df_codigos mediante un inner join usando 'codigo_iso'
# Nota: Si df_anio usa la columna 'pais' pero df_codigos también la tiene, podemos unir por 'pais' o renombrar si es necesario.
# Revisando el enunciado: la relación se hace típicamente por las claves compartidas.
# Si df_anio viene con las columnas ['pais', 'anio', 'indice', 'ranking'] y df_codigos con ['codigo_iso', 'pais'],
# podemos hacer el merge por 'pais'.
df = pd.merge(df_anio, df_codigos, on='pais', how='inner')

# Asegurar el orden estándar de las columnas solicitado en el diccionario
df = df[['codigo_iso', 'pais', 'anio', 'indice', 'ranking']]


KeyError: 'pais'



### 2. Exploración inicial del conjunto de datos

Una vez que hayas consolidado el DataFrame final `df`, realiza un análisis exploratorio básico respondiendo las siguientes preguntas:

#### **Estructura del DataFrame**

* ¿Cuántas **filas (observaciones)** contiene el conjunto de datos?
* ¿Cuántas **columnas** tiene el DataFrame?
* ¿Cuáles son los **nombres de las columnas**?
* ¿Qué **tipo de datos** tiene cada columna?
* ¿Hay columnas con un tipo de dato inesperado (por ejemplo, fechas como strings)?

#### **Resumen estadístico**

* Genera un resumen estadístico del conjunto de datos con `.describe()`.
  ¿Qué observas sobre los valores de `indice` y `ranking`?
* ¿Qué valores mínimo, máximo y promedio tiene la columna `indice`?
* ¿Qué países presentan los valores extremos en `indice` y `ranking`?

#### **Datos faltantes**

* ¿Cuántos valores nulos hay en cada columna?
* ¿Qué proporción de observaciones tienen valores faltantes?
* ¿Hay columnas con más del 30% de datos faltantes?

#### **Unicidad y duplicados**

* ¿Cuántos países distintos (`pais`) hay en el DataFrame?
* ¿Cuántos años distintos (`anio`) hay representados?
* ¿Existen filas duplicadas (exactamente iguales)? ¿Cuántas?

#### **Validación cruzada de columnas**

* ¿Hay inconsistencias entre el país (`pais`) y su código (`codigo_iso`)?
  (por ejemplo, un mismo código ISO asociado a más de un país)

> **Sugerencia**: Apoya tu análisis con funciones como `.info()`, `.nunique()`, `.isnull().sum()`, `.duplicated()`, `.value_counts()`, entre otras.



    

In [11]:
# FIXME
# Estructura del DataFrame
print("--- Estructura del DataFrame ---")
df.info()

# Resumen estadístico
print("\n--- Resumen Estadístico ---")
print(df.describe())

# Datos faltantes
print("\n--- Datos Faltantes por Columna ---")
nulos = df.isnull().sum()
proporcion_nulos = df.isnull().mean()
for col in df.columns:
    print(f"{col}: {nulos[col]} nulos ({proporcion_nulos[col]:.2%})")

# Unicidad y duplicados
print("\n--- Unicidad y Duplicados ---")
print(f"Países únicos: {df['pais'].nunique()}")
print(f"Años únicos: {df['anio'].nunique()}")
print(f"Filas exactamente duplicadas: {df.duplicated().sum()}")

# Validación cruzada
Inconsistencias = df.groupby('codigo_iso')['pais'].nunique()
print(f"Códigos ISO con más de un país asociado: {(inconsistencias > 1).sum()}")

--- Estructura del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Empty DataFrame

--- Resumen Estadístico ---


ValueError: Cannot describe a DataFrame without columns




### 3. Comparación regional: países latinoamericanos

En esta sección se busca identificar cuáles son los países de América Latina que han presentado los valores extremos del **Índice de Libertad de Prensa** en cada año observado.

> Recuerda que un menor puntaje en `indice` implica mayor libertad de prensa.

#### **Tareas:**

**a)** Utilizando un ciclo `for`, recorre cada año del conjunto de datos filtrado por países latinoamericanos, y determina para cada año:

* El país con el menor valor de `indice` (mayor libertad de prensa).
* El país con el mayor valor de `indice` (menor libertad de prensa).

**b)** Resuelve la misma tarea del punto anterior utilizando un enfoque vectorizado con `groupby`, sin usar ciclos explícitos.



#### **Lista de países latinoamericanos considerada:**

```python
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
           'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
           'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
           'USA', 'VEN']
```

> Puedes usar esta lista para filtrar el DataFrame final por la columna `codigo_iso`.



In [12]:
# respuesta
america = ['ARG', 'ATG', 'BLZ', 'BOL', 'BRA', 'CAN', 'CHL', 'COL', 'CRI',
       'CUB', 'DOM', 'ECU', 'GRD', 'GTM', 'GUY', 'HND', 'HTI', 'JAM',
       'MEX', 'NIC', 'PAN', 'PER', 'PRY', 'SLV', 'SUR', 'TTO', 'URY',
       'USA', 'VEN']

df_america =  pd.DataFrame() # FIX ME

### 4. Análisis anual del índice por país

En esta sección se busca analizar la evolución del **índice máximo** de libertad de prensa alcanzado por cada país a lo largo del tiempo.

#### **Tarea principal:**

* Construye una tabla dinámica (`pivot_table`) donde las **filas** correspondan a los países, las **columnas** a los años (`anio`) y los **valores** sean el `indice` máximo alcanzado por cada país en ese año.
* Asegúrate de reemplazar los valores nulos resultantes con `0`.

> **Hint**: Puedes utilizar el parámetro `fill_value=0` en `pd.pivot_table(...)`.



#### **Preguntas adicionales:**

**a)** ¿Qué país tiene el mayor valor de `indice` en toda la tabla resultante? ¿Y cuál tiene el menor (distinto de cero)?
**b)** ¿Qué años presentan en promedio los valores de `indice` más altos? ¿Y los más bajos?

> (Pista: usa `.mean(axis=0)` sobre la tabla pivot)

**c)** ¿Qué país muestra mayor **variabilidad** (diferencia entre su máximo y mínimo `indice` a lo largo del tiempo)?

> (Pista: aplica `.max(axis=1) - .min(axis=1)`)

**d)** ¿Existen países con índice constante a lo largo de todos los años registrados? ¿Cuáles?

**e)** ¿Qué países no tienen ningún dato (es decir, quedaron con todos los valores igual a 0)? ¿Podrías explicar por qué?





In [14]:
# FIX ME
# Tarea principal: Construir la tabla dinámica (pivot_table)
df_pivot = pd.pivot_table(
    df,
    values='indice',
    index='pais',
    columns='anio',
    aggfunc='max',
    fill_value=0
)

print("--- Tabla Dinámica Creada (Primeras filas) ---")
print(df_pivot.head())
print("\n-----------------------------------------------\n")

# Respuestas a preguntas adicionales:

# a) Mayor valor de índice global y menor valor (distinto de cero)
val_max_global = df_pivot.max().max()
pais_max_global = df_pivot.max(axis=1).idxmax()

# Para el mínimo distinto de cero, enmascaramos los ceros como NaN
df_pivot_no_zeros = df_pivot.mask(df_pivot == 0)
val_min_global = df_pivot_no_zeros.min().min()
pais_min_global = df_pivot_no_zeros.min(axis=1).idxmin()

print(f"a) Mayor índice global: {pais_max_global} ({val_max_global})")
print(f"   Menor índice global (de los válidos > 0): {pais_min_global} ({val_min_global})")

# b) Años en promedio con valores más altos y más bajos
promedio_por_anio = df_pivot_no_zeros.mean(axis=0) # axis=0 opera verticalmente (años)
anio_mas_alto = promedio_por_anio.idxmax()
anio_mas_bajo = promedio_por_anio.idxmin()
print(f"b) Año con promedio de índice más alto (menor libertad general): {anio_mas_alto}")
print(f"   Año con promedio de índice más bajo (mayor libertad general): {anio_mas_bajo}")

# c) País con mayor variabilidad (Máximo - Mínimo) a lo largo del tiempo
variabilidad = df_pivot_no_zeros.max(axis=1) - df_pivot_no_zeros.min(axis=1)
pais_mas_variable = variabilidad.idxmax()
print(f"c) País con mayor variabilidad histórica de prensa: {pais_mas_variable} (Diferencia de {variabilidad.max():.2f})")

# d) Países con índice constante (Variabilidad igual a 0)
paises_constantes = variabilidad[variabilidad == 0].index.tolist()
print(f"d) Países con índice constante a lo largo de los años registrados: {paises_constantes}")

# e) Países sin ningún dato (quedaron completamente en cero en la tabla original)
paises_sin_datos = df_pivot.index[df_pivot.sum(axis=1) == 0].tolist()
print(f"e) Países que no registraron ningún dato (quedaron en 0): {paises_sin_datos}")
print("   Explicación: Esto ocurre porque dichos países no figuraban en los archivos temporales de mediciones")
print("   o sus nombres no coincidieron exactamente al aplicar la fusión (merge) inicial.")


KeyError: 'indice'